# 🧠 Schizophrenia Recognition GNN+CEFAM — Tier 4 on Google Colab

Notebook này hướng dẫn chạy huấn luyện mô hình **Tier 4 (Spatiotemporal GNN + CEFAM)** sử dụng GPU miễn phí trên Google Colab từ kho lưu trữ GitHub của bạn.

> **Mẹo Tối Ưu Tốc Độ:** Vì bạn đã chạy xong Tier 1, 2, 3 ở local, bạn **KHÔNG CẦN** tải lên thư mục `EMS` thô (nặng hàng trăm MB) nữa. Thay vào đó, hãy dùng **Cách 1: Fast Track** dưới đây (chỉ tải lên các file đặc trưng đã xử lý ~7MB) để chạy ngay Tier 4.

## 🛠 Bước 1: Cấu hình Môi trường & Clone Repository
Chọn **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU** trước khi chạy.

In [ ]:
# 1. Clone repository từ GitHub của bạn
!git clone https://github.com/haimayoi/eye-movement-based-schizophrenia-recognition.git
%cd eye-movement-based-schizophrenia-recognition

In [ ]:
# 2. Cài đặt các thư viện cần thiết (bao gồm PyTorch Geometric)
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm

## 📂 Bước 2: Chuẩn bị Dữ liệu (Chọn 1 trong 2 cách)

### Cách 1: FAST TRACK (Khuyên Dùng - Chỉ tải lên 4 file kết quả đã chạy ở local ~7MB)
Nén và tải các file sau lên Colab (kéo thả trực tiếp vào bảng điều khiển Files của Colab hoặc mount Drive):
- `data/metadata/stimulus_categories.csv`
- `data/processed/clean_fixations.parquet`
- `data/processed/features_stimulus_level.csv`
- `data/processed/features_subject_level.csv`

Bạn nén thư mục `data` trên máy của bạn thành file `data_processed.zip` rồi chạy lệnh giải nén dưới đây:

In [ ]:
# Chạy lệnh này nếu bạn upload file nén data_processed.zip lên thư mục gốc /content/
# !unzip -o /content/data_processed.zip -d /content/eye-movement-based-schizophrenia-recognition/
# print("Đã giải nén dữ liệu preprocessed thành công! Bạn có thể bỏ qua Bước 3 và nhảy thẳng sang Bước 4.")

### Cách 2: FULL RUN (Nếu bạn muốn chạy lại toàn bộ Tier 1, 2, 3 trên Colab)
Upload thư mục `EMS` thô lên Google Drive rồi tạo liên kết:

In [ ]:
# Chỉ chạy nếu sử dụng Cách 2
# from google.colab import drive
# drive.mount('/content/drive')
# !ln -s "/content/drive/MyDrive/EMS" ./EMS

## 🚀 Bước 3: Chạy Tiền xử lý & Trích xuất Đặc trưng (Chỉ chạy nếu chọn Cách 2)

In [ ]:
# Chỉ chạy nếu sử dụng Cách 2
# !python -m src.utils.generate_category_map
# !python -m src.tier1_preprocessing.preprocess
# !python -m src.tier2_features.stimulus_features
# !python -m src.tier2_features.subject_aggregator

## 🔷 Bước 4: Xây dựng Đồ thị Scanpath (PyG Graphs)
Tác vụ này chuyển đổi các chuỗi mắt nhãn cầu thành đồ thị GNN và tự động sinh dummy RINet features 1056-dim để khớp nối luồng.

In [ ]:
!python -m src.tier4_advanced.graph_builder

## 🔥 Bước 5: Huấn luyện Mô hình GNN + CEFAM Hybrid Model
Sử dụng GPU T4 của Google Colab để huấn luyện mô hình hybrid với 4-Fold CV.

In [ ]:
!python scripts/train_tier4.py --config configs/cefam_config.yaml

## 📥 Bước 6: Lưu kết quả huấn luyện
Sao chép các checkpoints và file kết quả thu được về Google Drive để lưu trữ lâu dài.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Tạo thư mục lưu kết quả trên Google Drive
!mkdir -p "/content/drive/MyDrive/SZ_Recognition_Results/checkpoints"

# Sao chép các tệp checkpoint (.pt) và file kết quả summary (.json, .csv)
!cp -r results/checkpoints/* "/content/drive/MyDrive/SZ_Recognition_Results/checkpoints/"
!cp results/cefam_results_summary.json "/content/drive/MyDrive/SZ_Recognition_Results/"
!cp results/cefam_subject_val_predictions.csv "/content/drive/MyDrive/SZ_Recognition_Results/"